
# 00 — Dataset Exploration and Validation

## Objective

This notebook validates the Digikala dataset before developing the Product Discovery and Review-Based QA modules.

The initial inspection focuses on:

- Locating the products and comments CSV files.
- Validating file availability and integrity.
- Inspecting dataset schemas and representative samples.
- Identifying potential data quality issues.

The analysis uses small samples to avoid loading millions of records into memory.


In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "pyproject.toml").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

load_dotenv(project_root / ".env", override=True)

data_dir = os.getenv("DIGIKALA_DATA_DIR")

if not data_dir:
    raise EnvironmentError("DIGIKALA_DATA_DIR is not configured.")

data_root = Path(data_dir).expanduser().resolve()

if not data_root.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {data_root}")

file_names = {
    "products": "digikala-products.csv",
    "comments": "digikala-comments.csv"
}

data_paths = {}

for name, filename in file_names.items():
    matches = list(data_root.rglob(filename))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename}, found {len(matches)}."
        )

    path = matches[0]

    with path.open("rb") as file:
        header = file.read(128)

    if b"git-lfs.github.com/spec" in header:
        raise RuntimeError(f"Git LFS content not downloaded: {filename}")

    data_paths[name] = path

    print(f"{name.upper()}")
    print(f"Path: {path}")
    print(f"Size: {path.stat().st_size / 1024**2:.2f} MB")
    print()


PRODUCTS
Path: /home/mm/Projects/RAG/Digikala_comments_products/digikala-products.csv
Size: 246.75 MB

COMMENTS
Path: /home/mm/Projects/RAG/Digikala_comments_products/digikala-comments.csv
Size: 1219.30 MB




## Initial Schema and Data Quality Inspection

A small sample of each CSV file is loaded to inspect column names, data types, missing values, and sample records.

These observations are preliminary and do not represent statistics for the complete dataset. Full-dataset validation will be performed separately where necessary.


In [2]:
samples = {
    name: pd.read_csv(path, nrows=1000, low_memory=False)
    for name, path in data_paths.items()
}

for name, df in samples.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} DATASET")
    print(f"{'=' * 60}")

    print(f"\nSample shape: {df.shape}")

    print("\nColumn information:")
    print(df.dtypes.to_string())

    print("\nMissing values (%):")
    print(
        df.isna()
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
        .to_string()
    )

    print("\nSample records:")
    display(df.head(3))



PRODUCTS DATASET

Sample shape: (1000, 12)

Column information:
id                      int64
title_fa                  str
Rate                    int64
Rate_cnt                int64
Category1                 str
Category2                 str
Brand                     str
Price                   int64
Seller                    str
Is_Fake                  bool
min_price_last_month    int64
sub_category              str

Missing values (%):
Category2               3.1
Seller                  0.1
id                      0.0
title_fa                0.0
Rate_cnt                0.0
Rate                    0.0
Brand                   0.0
Category1               0.0
Price                   0.0
Is_Fake                 0.0
min_price_last_month    0.0
sub_category            0.0

Sample records:


,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,90,4,آبسلانگ,NaN,متفرقه,634800,سلامت ساز راد,False,0,beauty
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,84,217,آبسلانگ,NaN,متفرقه,818800,مهر افزون,False,0,beauty
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,88,14,آبسلانگ,NaN,متفرقه,920000,یانگوم,False,0,beauty



COMMENTS DATASET

Sample shape: (1000, 15)

Column information:
id                         int64
title                        str
body                         str
created_at                   str
rate                     float64
recommendation_status        str
is_buyer                    bool
product_id                 int64
advantages                   str
disadvantages                str
likes                      int64
dislikes                   int64
seller_title                 str
seller_code                  str
true_to_size_rate            str

Missing values (%):
true_to_size_rate        99.5
disadvantages            89.0
advantages               81.9
title                    38.6
recommendation_status    11.1
seller_title              3.9
seller_code               3.9
is_buyer                  0.0
rate                      0.0
created_at                0.0
body                      0.0
id                        0.0
product_id                0.0
likes                     0.0

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,0,0,دیجی‌کالا,5A52N,NaN
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],1,0,دیجی‌کالا,5A52N,NaN
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,NaN,NaN,0,0,آرالیا بیوتی,ADM47,NaN



## Full-Dataset Review Statistics

This section performs a memory-efficient scan of the comments dataset using chunked processing.

The objectives are to:

- Count the total number of reviews.
- Measure the distribution of recommendation labels.
- Identify products with sufficient review coverage.
- Detect missing or empty review bodies.
- Prepare candidates for a representative development dataset.

Only the required columns are loaded to reduce memory consumption.


In [3]:
from collections import Counter

review_counts = Counter()
label_counts = Counter()

total_comments = 0
empty_bodies = 0

for chunk in pd.read_csv(
    data_paths["comments"],
    usecols=["product_id", "body", "recommendation_status"],
    chunksize=100_000
):
    total_comments += len(chunk)

    review_counts.update(
        chunk["product_id"].value_counts().to_dict()
    )

    label_counts.update(
        chunk["recommendation_status"]
        .fillna("MISSING")
        .value_counts()
        .to_dict()
    )

    empty_bodies += (
        chunk["body"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

print(f"Total reviews: {total_comments:,}")
print(f"Unique reviewed products: {len(review_counts):,}")
print(f"Empty review bodies: {empty_bodies:,}")

label_distribution = pd.DataFrame(
    [
        {
            "label": label,
            "count": count,
            "percentage": round(count / total_comments * 100, 2)
        }
        for label, count in label_counts.items()
    ]
).sort_values("count", ascending=False)

display(label_distribution)


Total reviews: 6,156,289
Unique reviewed products: 331,599
Empty review bodies: 637


,label,count,percentage
0,recommended,4162839,67.62
1,MISSING,894426,14.53
2,no_idea,594121,9.65
3,not_recommended,504903,8.20



## Selecting Product Candidates for Development

Products with substantial review coverage are identified and matched with their catalog metadata.

To support meaningful testing across different product types, the selection includes multiple product categories rather than focusing only on the most frequently reviewed items.

The resulting candidates will be used to build a small, reproducible development dataset for Product Discovery and Review-Based QA.


In [4]:
top_product_ids = {
    product_id
    for product_id, count in review_counts.most_common(1500)
}

candidate_chunks = []

for chunk in pd.read_csv(
    data_paths["products"],
    chunksize=100_000
):
    matching = chunk.loc[
        chunk["id"].isin(top_product_ids)
    ].copy()

    if not matching.empty:
        candidate_chunks.append(matching)

if not candidate_chunks:
    raise ValueError("No matching products found in the catalog.")

candidate_products = pd.concat(
    candidate_chunks,
    ignore_index=True
)

candidate_products["review_count"] = (
    candidate_products["id"].map(review_counts)
)

candidate_products = candidate_products.drop_duplicates(
    subset=["id"]
)

selected_products = (
    candidate_products
    .sort_values("review_count", ascending=False)
    .groupby("sub_category", dropna=False)
    .head(2)
    .sort_values("review_count", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

display(
    selected_products[
        [
            "id",
            "title_fa",
            "Brand",
            "Price",
            "sub_category",
            "Rate",
            "Rate_cnt",
            "review_count"
        ]
    ]
)


,id,title_fa,Brand,Price,sub_category,Rate,Rate_cnt,review_count
0,4095187,برس مژه و ابرو کد BR01ABO,متفرقه,83000,beauty,82,1487,400
1,3331597,برس ریمل مدل LM02 مجموعه 2 عددی,متفرقه,79000,beauty,84,587,400
2,7832807,عینک آفتابی مدل GANG,متفرقه,640000,clothe,82,1167,209
3,2755588,دستمال مرطوب بانیو مدل آنتی باکتریال بسته 96 عددی,بانیو,483900,toys and kids,86,719,209
4,6874012,دستمال مرطوب طرح یونیکورن کد 01 بسته 30 عددی,متفرقه,417000,toys and kids,80,331,204
5,9598793,کفش پیاده روی زنانه مدل ال استار کد AZ2,متفرقه,1600000,clothe,76,196,204
6,206504,کتاب خانه درختی 13 طبقه اثر اندی گریفیتس,هوپا,1089000,book & stationary & art,88,669,200
7,3467059,کتاب دختری در اتاق طبقه‌ی سوم اثر مری داونینگ ...,انتشارات پرتقال,881200,book & stationary & art,90,279,200



## Building a Reproducible Development Dataset

A development dataset is created by combining a random sample of reviewed products with high-review product candidates.

All available reviews for the selected products are extracted using chunked processing. This ensures that Product Discovery and Review-Based QA operate on the same product catalog and that review evidence can be traced back to its source.

The sampling process uses a fixed random seed for reproducibility. The resulting subset is intended for development and integration testing, not as a replacement for full-dataset evaluation.


In [5]:
import numpy as np

rng = np.random.default_rng(42)

reviewed_ids = np.fromiter(
    review_counts.keys(),
    dtype=np.int64
)

random_ids = set(
    rng.choice(
        reviewed_ids,
        size=min(5000, len(reviewed_ids)),
        replace=False
    ).tolist()
)

featured_products = (
    candidate_products
    .sort_values("review_count", ascending=False)
    .groupby("sub_category", dropna=False)
    .head(15)
)

target_ids = random_ids | set(featured_products["id"].tolist())

product_parts = []

for chunk in pd.read_csv(
    data_paths["products"],
    chunksize=100_000
):
    matches = chunk.loc[chunk["id"].isin(target_ids)]

    if not matches.empty:
        product_parts.append(matches)

if not product_parts:
    raise ValueError("No matching products found.")

products_dev = (
    pd.concat(product_parts, ignore_index=True)
    .drop_duplicates(subset=["id"])
    .sort_values("id")
    .reset_index(drop=True)
)

matched_ids = set(products_dev["id"].tolist())

review_parts = []

for chunk in pd.read_csv(
    data_paths["comments"],
    chunksize=100_000,
    low_memory=False
):
    matches = chunk.loc[
        chunk["product_id"].isin(matched_ids)
    ].copy()

    if not matches.empty:
        matches["body"] = matches["body"].fillna("").str.strip()
        matches = matches.loc[matches["body"].ne("")]

        if not matches.empty:
            review_parts.append(matches)

if not review_parts:
    raise ValueError("No matching reviews found.")

reviews_dev = (
    pd.concat(review_parts, ignore_index=True)
    .drop_duplicates(subset=["id"])
    .sort_values(["product_id", "id"])
    .reset_index(drop=True)
)

assert reviews_dev["product_id"].isin(products_dev["id"]).all()

sample_dir = project_root / "data" / "sample"
sample_dir.mkdir(parents=True, exist_ok=True)

products_dev.to_csv(
    sample_dir / "products_dev.csv",
    index=False
)

reviews_dev.to_csv(
    sample_dir / "reviews_dev.csv",
    index=False
)

print(f"Selected products: {len(products_dev):,}")
print(f"Selected reviews: {len(reviews_dev):,}")
print(f"Reviewed products: {reviews_dev['product_id'].nunique():,}")
print(f"Product categories: {products_dev['sub_category'].nunique()}")
print(f"Development files saved to: {sample_dir}")

display(
    products_dev["sub_category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="product_count")
)


Selected products: 5,060
Selected reviews: 103,174
Reviewed products: 5,060
Product categories: 6
Development files saved to: /home/mm/Projects/ai-bootcamp-project-3/data/sample


,category,product_count
0,clothe,1971
1,book & stationary & art,1337
2,beauty,1043
3,toys and kids,489
4,travel,117
5,rural goods,103
